## CSE 151B Competition — Optimized Notebook (v2)

| # | Change | Impact |
|---|---|---|
| 1 | **LaTeX repair** — restore `\frac`, `\infty`, `\int`, etc. before sending to model | Highest ROI |
| 2 | **Free-form majority voting** — extract `\boxed{}` from all 5 samples, normalize via sympy, pick modal answer | Large (67% of dataset) |
| 3 | **Verifier pass** — filter voted answers through sympy parse check before accepting | Accuracy |
| 4 | **MCQ logprobs** — no-think constrained pass reads P(letter) directly; much more calibrated than text votes | +5–10 pp MCQ |
| 5 | **Adaptive N** — N=3 initial MCQ; expand to N=9 only for no-consensus questions | Efficient compute |
| 6 | **Better extraction fallback** — returns `""` (abstain) instead of last random capital letter | Reduces noise |
| 7 | **MCQ stop sequences + max_tokens=4000** — prevents post-answer rambling | Speed + accuracy |
| 8 | **MCQ prompt**: "After `</think>`, output exactly `\boxed{X}`" | Format compliance |
| 9 | **Free-form multi-blank few-shot + max_tokens=4096** | Format anchoring |

## 1. Environment Setup

Same as starter. Comment out the install block after first run, then restart the kernel.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create venv and install — use full path since PATH isn't updated yet
# !~/.local/bin/uv venv .venv --seed
# !~/.local/bin/uv pip install --python .venv/bin/python sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")

In [2]:
# Activate venv — run every time
!source ./.venv/bin/activate

## 2. Imports & Configuration

Key changes vs starter:
- `MAX_TOKENS = 12000` — fits within `max_model_len=16384` (leaves ~4k for prompt)
- `N_SAMPLES = 5` — enables majority voting
- `N_QUESTIONS = 10` — run first 10 questions for quick evaluation

In [3]:
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Optional

import sympy as sp
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID         = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID           = "0"
DATA_PATH        = "data/public.jsonl"
OUTPUT_PATH      = "results/optimized_results.jsonl"
MAX_TOKENS_MCQ   = 4000   # #7: capped — thinking models ramble after settling
MAX_TOKENS_FREE  = 4096   # #9: constant budget for free-form
N_SAMPLES_INIT   = 3      # #5: initial samples per MCQ question
N_SAMPLES_EXPAND = 9      # #5: expanded samples for no-consensus MCQ questions
N_FREE_SAMPLES   = 5      # samples for free-form majority voting (#2)
N_QUESTIONS      = 10     # first N questions to evaluate

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 3. Load the Dataset

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")
print(f"Will evaluate first {N_QUESTIONS} questions.")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)
Will evaluate first 10 questions.

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. LaTeX Repair (Fix #1)

The dataset stores `frac{a}{b}`, `infty`, `int`, `sin`, etc. **without backslashes**.
The model is being handed malformed LaTeX — fixing this is the single highest-ROI change.

A regex pass restores the backslashes before building any prompt.

In [ ]:
# Fix #1: restore missing backslashes before LaTeX commands.
# Pattern: word-boundary match on the bare command, negative lookbehind to skip
# already-correct \frac etc.  "times" and "pi" are safe in all-math questions.
_LATEX_SUBS = [
    (r'(?<!\\)\bfrac\b',    r'\\frac'),
    (r'(?<!\\)\binfty\b',   r'\\infty'),
    (r'(?<!\\)\bint\b',     r'\\int'),
    (r'(?<!\\)\bsum\b',     r'\\sum'),
    (r'(?<!\\)\bprod\b',    r'\\prod'),
    (r'(?<!\\)\blim\b',     r'\\lim'),
    (r'(?<!\\)\bsqrt\b',    r'\\sqrt'),
    (r'(?<!\\)\bsin\b',     r'\\sin'),
    (r'(?<!\\)\bcos\b',     r'\\cos'),
    (r'(?<!\\)\btan\b',     r'\\tan'),
    (r'(?<!\\)\bcot\b',     r'\\cot'),
    (r'(?<!\\)\bsec\b',     r'\\sec'),
    (r'(?<!\\)\bcsc\b',     r'\\csc'),
    (r'(?<!\\)\blog\b',     r'\\log'),
    (r'(?<!\\)\bln\b',      r'\\ln'),
    (r'(?<!\\)\bexp\b',     r'\\exp'),
    (r'(?<!\\)\bpi\b',      r'\\pi'),
    (r'(?<!\\)\btheta\b',   r'\\theta'),
    (r'(?<!\\)\balpha\b',   r'\\alpha'),
    (r'(?<!\\)\bbeta\b',    r'\\beta'),
    (r'(?<!\\)\bgamma\b',   r'\\gamma'),
    (r'(?<!\\)\bdelta\b',   r'\\delta'),
    (r'(?<!\\)\bsigma\b',   r'\\sigma'),
    (r'(?<!\\)\blambda\b',  r'\\lambda'),
    (r'(?<!\\)\bmu\b',      r'\\mu'),
    (r'(?<!\\)\bepsilon\b', r'\\epsilon'),
    (r'(?<!\\)\bphi\b',     r'\\phi'),
    (r'(?<!\\)\bomega\b',   r'\\omega'),
    (r'(?<!\\)\bcdot\b',    r'\\cdot'),
    (r'(?<!\\)\bpm\b',      r'\\pm'),
    (r'(?<!\\)\bleq\b',     r'\\leq'),
    (r'(?<!\\)\bgeq\b',     r'\\geq'),
    (r'(?<!\\)\bneq\b',     r'\\neq'),
    (r'(?<!\\)\btimes\b',   r'\\times'),
    (r'(?<!\\)\bpartial\b', r'\\partial'),
    (r'(?<!\\)\bnabla\b',   r'\\nabla'),
]

_COMPILED_SUBS = [(re.compile(p), r) for p, r in _LATEX_SUBS]

def repair_latex(text: str) -> str:
    """Restore missing backslashes before LaTeX commands."""
    for pattern, replacement in _COMPILED_SUBS:
        text = pattern.sub(replacement, text)
    return text

# Smoke-test on the MCQ sample from the dataset
_test = "int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds"
print("Before:", _test)
print("After: ", repair_latex(_test))

## 5. Prompt Construction (Fixes #8, #9)

- **Fix #8 (MCQ)**: "After `</think>`, output exactly `\boxed{X}`" — works with Qwen3's thinking block structure instead of fighting it
- **Fix #9 (free-form)**: Two worked examples covering 2-answer and 3-answer cases; explicit single-`\boxed{}` rule; `repair_latex` applied to all question text

In [5]:
# Fix #8: MCQ prompt tuned for thinking model.
# The chain-of-thought lives in <think>...</think>. We don't suppress it —
# we just be unambiguous about what must follow </think>.
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Think carefully through each option. "
    "After </think>, output exactly one line: \\boxed{X} "
    "where X is the single correct letter from the options (A, B, C, …). "
    "Nothing else after </think>."
)

# Fix #9: free-form prompt with two few-shot examples (2-answer and 3-answer)
# and an explicit single-\boxed{} rule.
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Use exactly one \\boxed{} containing all final answers separated by commas in order. "
    "Do NOT use multiple \\boxed{} blocks.\n\n"
    "Example — two answers:\n"
    "Problem: Find the roots of x^2 - 5x + 6 = 0.\n"
    "Solution: (x-2)(x-3)=0, so x=2 or x=3.\n"
    "\\boxed{2, 3}\n\n"
    "Example — three answers:\n"
    "Problem: A right triangle has legs a=3, b=4. Find hypotenuse c, perimeter P, and area A.\n"
    "Solution: c=5, P=3+4+5=12, A=½·3·4=6.\n"
    "\\boxed{5, 12, 6}"
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt); applies LaTeX repair to question text."""
    question = repair_latex(question)           # Fix #1: repair before sending
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{lbl}. {repair_latex(opt.strip())}" for lbl, opt in zip(labels, options)
        )
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} system prompt ──")
    print(sys_p)
    print(f"\n── {label} user prompt (first 300 chars) ──")
    print(usr_p[:300], "...\n")

## 6. Load Model with vLLM

Four separate `SamplingParams` objects cover the different generation strategies:

| Params | Used for | Key settings |
|---|---|---|
| `mcq_sampling_init` | MCQ initial pass | N=3, max_tokens=4000, stop sequences (#7) |
| `mcq_sampling_expand` | MCQ expansion for no-consensus (#5) | N=9, same stops |
| `free_sampling` | Free-form voting (#2) | N=5, max_tokens=4096 |
| `logprob_sampling` | MCQ calibration pass (#4) | N=1, max_tokens=10, logprobs=20, no thinking |

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=True,   # Fix #6 (original): cache shared system-prompt KV
    gpu_memory_utilization=0.50,  # conservative: GPU is shared; ~11.8 GiB requested
    max_model_len=16384,
    trust_remote_code=True,
    enforce_eager=True,           # skip CUDA graph warmup to avoid OOM
    max_num_seqs=64,
)

_BASE = dict(temperature=0.7, top_p=0.8, top_k=20, min_p=0.0,
             presence_penalty=0.0, repetition_penalty=1.0)

# Fix #7: stop sequences keep thinking models from second-guessing after the answer
_MCQ_STOPS = ["</answer>", "\n\nProblem", "\n\nQuestion"]

# Fix #5: two MCQ configs — initial N=3, expanded N=9
mcq_sampling_init   = SamplingParams(n=N_SAMPLES_INIT,   max_tokens=MAX_TOKENS_MCQ,
                                     stop=_MCQ_STOPS, **_BASE)
mcq_sampling_expand = SamplingParams(n=N_SAMPLES_EXPAND, max_tokens=MAX_TOKENS_MCQ,
                                     stop=_MCQ_STOPS, **_BASE)

# Fix #2: free-form always uses 5 samples for majority voting
free_sampling = SamplingParams(n=N_FREE_SAMPLES, max_tokens=MAX_TOKENS_FREE, **_BASE)

# Fix #4: logprobs calibration pass — near-greedy, no thinking, single token budget
logprob_sampling = SamplingParams(
    n=1, max_tokens=10,
    temperature=0.01, top_p=1.0, top_k=-1,
    logprobs=20,
)

print("Model and sampling params ready.")

## 7. Generate Responses

Four generation passes in order:
1. **Free-form**: N=5 samples for all free-form questions
2. **MCQ initial**: N=3 samples for all MCQ questions
3. **MCQ adaptive expansion** (Fix #5): questions with no majority (< 2/3 votes agreeing) are re-run with N=9
4. **MCQ logprobs calibration** (Fix #4): a cheap no-think constrained pass reads P(letter) directly over A–J

In [7]:
def make_prompt(item: dict) -> str:
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

def make_calib_prompt(item: dict) -> str:
    """Fix #4: no-think constrained prompt for MCQ logprobs calibration pass."""
    _, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system",
          "content": "Select the correct answer letter (A–J). Output only the letter, nothing else."},
         {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=False,   # Qwen3-specific: suppress the <think> block
    )

# ── Split subset by question type ─────────────────────────────────────────────
subset       = data[:N_QUESTIONS]
mcq_indices  = [i for i, d in enumerate(subset) if d.get("options")]
free_indices = [i for i, d in enumerate(subset) if not d.get("options")]

mcq_prompts  = [make_prompt(subset[i]) for i in mcq_indices]
free_prompts = [make_prompt(subset[i]) for i in free_indices]

# ── Pass 1: Free-form (N=5) ────────────────────────────────────────────────────
print(f"[1/4] Free-form: {len(free_prompts)} q × N={N_FREE_SAMPLES}...")
free_outputs = llm.generate(free_prompts, free_sampling) if free_prompts else []

# ── Pass 2: MCQ initial (N=3) ─────────────────────────────────────────────────
print(f"[2/4] MCQ initial: {len(mcq_prompts)} q × N={N_SAMPLES_INIT}...")
mcq_outputs = list(llm.generate(mcq_prompts, mcq_sampling_init)) if mcq_prompts else []

# ── Pass 3: Adaptive MCQ expansion (Fix #5) ───────────────────────────────────
# A question has "consensus" if its top vote has ≥ 2 out of N_SAMPLES_INIT votes.
def has_consensus(out) -> bool:
    votes = [v for v in (_extract_letter_fast(o.text) for o in out.outputs) if v]
    return bool(votes) and Counter(votes).most_common(1)[0][1] >= 2

def _extract_letter_fast(text: str) -> str:
    m = re.search(r'\\boxed\{([A-Ja-j])\}', text)
    return m.group(1).upper() if m else ""

no_consensus_local = [li for li, out in enumerate(mcq_outputs) if not has_consensus(out)]

if no_consensus_local:
    print(f"[3/4] Expanding {len(no_consensus_local)} no-consensus MCQ to N={N_SAMPLES_EXPAND}...")
    expand_outs = llm.generate(
        [mcq_prompts[li] for li in no_consensus_local], mcq_sampling_expand
    )
    for k, li in enumerate(no_consensus_local):
        mcq_outputs[li] = expand_outs[k]   # replace with full expanded output
else:
    print("[3/4] All MCQ reached consensus — no expansion needed.")

# ── Pass 4: MCQ logprobs calibration (Fix #4) ─────────────────────────────────
print(f"[4/4] MCQ logprobs calibration: {len(mcq_prompts)} q...")
calib_prompts = [make_calib_prompt(subset[i]) for i in mcq_indices]
calib_outputs = list(llm.generate(calib_prompts, logprob_sampling)) if calib_prompts else []

print("\nGeneration complete.")
print(f"  MCQ: {len(mcq_outputs)} q | Free-form: {len(free_outputs)} q")

## 8. Score Responses

- **Fix #6**: `extract_letter` abstains (`""`) instead of guessing a random capital — restricts fallback to option-marker patterns only
- **Fix #4**: MCQ uses logprobs calibration letter as primary signal; text majority vote as fallback
- **Fix #2**: Free-form uses majority vote over `\boxed{}` content extracted from all 5 samples, normalized via sympy
- **Fix #3**: Verifier filters the voted answer through sympy parse before accepting

In [8]:
# ── Fix #6: letter extraction — no random-capital fallback ────────────────────
_ANSWER_PATTERNS = [
    r"\\boxed\{([A-Ja-j])\}",
    r"(?:the\s+)?answer\s+is\s+[\(\[]?([A-J])[\)\]]?",
    r"(?:correct\s+)?answer\s*[:\-]\s*[\(\[]?([A-J])[\)\]]",
    r"(?:therefore|thus|so),?\s+(?:the\s+)?(?:answer|choice|option)\s+is\s+[\(\[]?([A-J])[\)\]]",
    r"option\s+([A-J])\s+is\s+correct",
    r"select\s+(?:option\s+)?([A-J])\b",
    r"choose\s+(?:option\s+)?([A-J])\b",
]
_OPTION_MARKER = re.compile(r"^([A-J])[.)]\s", re.MULTILINE)

def extract_letter(text: str) -> str:
    for pat in _ANSWER_PATTERNS:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
    # Restricted fallback: letters appearing as option markers in the text
    markers = _OPTION_MARKER.findall(text)
    if markers:
        return markers[-1].upper()
    return ""   # abstain rather than guess


# ── Fix #4: MCQ logprobs calibration scoring ──────────────────────────────────
letter_token_ids = {}
for _ch in "ABCDEFGHIJ":
    _ids = tokenizer.encode(_ch, add_special_tokens=False)
    if _ids:
        letter_token_ids[_ch] = _ids[0]

def score_mcq_logprobs(calib_out, options_count: int) -> tuple[str, float]:
    """Return (best_letter, confidence) from the calibration pass."""
    gen_text = calib_out.outputs[0].text.strip()
    # Fast path: first character is a valid option letter
    valid = [chr(65 + i) for i in range(options_count)]
    if gen_text and gen_text[0].upper() in valid:
        return gen_text[0].upper(), 1.0
    # Logprobs path
    lps = calib_out.outputs[0].logprobs
    if lps:
        first_pos = lps[0]   # dict[token_id -> Logprob]
        best_letter, best_lp = "", float("-inf")
        for letter in valid:
            tid = letter_token_ids.get(letter)
            if tid and tid in first_pos:
                lp = first_pos[tid].logprob
                if lp > best_lp:
                    best_lp, best_letter = lp, letter
        if best_letter:
            return best_letter, best_lp
    # Final fallback: extract from generated text
    return extract_letter(gen_text), 0.0

def score_mcq(response_set, gold_letter, calib_out=None, options_count=10):
    """MCQ: logprobs primary, text majority vote fallback."""
    meta = {}
    calib_letter = ""
    if calib_out is not None:
        calib_letter, conf = score_mcq_logprobs(calib_out, options_count)
        meta.update(calib_letter=calib_letter, calib_conf=round(conf, 4))

    votes      = [extract_letter(r) for r in response_set]
    votes      = [v for v in votes if v]
    vote_letter = Counter(votes).most_common(1)[0][0] if votes else ""
    meta.update(votes=votes, vote_letter=vote_letter)

    pred        = calib_letter if calib_letter else vote_letter
    meta["pred"] = pred
    gold_upper  = gold_letter.strip().upper()
    return pred == gold_upper, pred, meta


# ── Fix #2 + #3: free-form majority voting with verifier ─────────────────────
def extract_boxed_content(text: str) -> str:
    """Extract content from the LAST \\boxed{} in text."""
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    return matches[-1].strip() if matches else ""

def normalize_ans(s: str) -> str:
    """Normalize for voting comparison: sympy-simplify numerics, else lowercase."""
    s = s.strip()
    try:
        val = sp.sympify(s, evaluate=True)
        if val.is_number:
            return str(round(float(val), 6))
        return str(val)
    except Exception:
        return s.lower().replace(" ", "")

def verify_answer(answer: str) -> bool:
    """Fix #3: lightweight verifier — checks the answer parses as valid sympy.
    Non-sympy answers (text) are accepted as-is if non-empty."""
    if not answer:
        return False
    try:
        sp.sympify(answer, evaluate=True)
        return True
    except Exception:
        return len(answer.strip()) > 0

def vote_free_form(response_set: list[str]) -> tuple[str, str]:
    """Extract \\boxed{}, normalize, majority-vote; prefer verified answers (#3)."""
    raw  = [extract_boxed_content(r) for r in response_set]
    raw  = [r for r in raw if r]
    if not raw:
        return "", ""
    norms  = [normalize_ans(r) for r in raw]
    counts = Counter(norms)
    for modal_norm, _ in counts.most_common():
        for r, n in zip(raw, norms):
            if n == modal_norm and verify_answer(r):
                return r, modal_norm
    # Fall back: most common regardless of verify
    modal_norm = counts.most_common(1)[0][0]
    for r, n in zip(raw, norms):
        if n == modal_norm:
            return r, modal_norm
    return raw[0], norms[0]


# ── Load judger ───────────────────────────────────────────────────────────────
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

# ── Build response maps keyed by subset index ─────────────────────────────────
mcq_response_map  = {mcq_indices[li]:  [o.text.strip() for o in out.outputs]
                     for li, out in enumerate(mcq_outputs)}
free_response_map = {free_indices[li]: [o.text.strip() for o in out.outputs]
                     for li, out in enumerate(free_outputs)}
calib_map         = {mcq_indices[li]:  calib_outputs[li]
                     for li in range(len(calib_outputs))}

# ── Score ─────────────────────────────────────────────────────────────────────
results = []
for idx, item in tqdm(enumerate(subset), total=len(subset), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        response_set  = mcq_response_map.get(idx, [])
        options_count = len(item.get("options", []))
        correct, pred, meta = score_mcq(
            response_set, str(gold),
            calib_out=calib_map.get(idx),
            options_count=options_count,
        )
        response = response_set[0] if response_set else ""
    else:
        response_set         = free_response_map.get(idx, [])
        voted_answer, modal  = vote_free_form(response_set)
        gold_list            = gold if isinstance(gold, list) else [gold]
        response             = response_set[0] if response_set else ""
        meta                 = {"voted_answer": voted_answer, "modal_norm": modal}
        if voted_answer:
            try:
                correct = judger.auto_judge(
                    pred=voted_answer, gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except Exception:
                correct = False
        else:
            correct = False

    results.append({
        "id": item.get("id"), "is_mcq": is_mcq,
        "gold": gold, "response": response, "correct": correct,
        **meta,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

In [9]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 55)
print("EVALUATION RESULTS (first 10 questions)")
print("=" * 55)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 55)

# MCQ: show calibration vs vote agreement
if mcq_res:
    print("\n── MCQ decision breakdown ──")
    for r in mcq_res:
        calib  = r.get("calib_letter", "?")
        vote   = r.get("vote_letter",  "?")
        pred   = r.get("pred",         "?")
        votes  = r.get("votes",        [])
        mark   = "✓" if r["correct"] else "✗"
        agree  = "✓ agree" if calib == vote else "✗ DISAGREE"
        print(f"  id={r['id']:4d}  gold={r['gold']}  "
              f"calib={calib}  votes={votes}→{vote}  pred={pred}  [{agree}] {mark}")

# Free-form: show voted answer vs gold
if free_res:
    print("\n── Free-form voted answers ──")
    for r in free_res:
        voted = r.get("voted_answer", "?")
        mark  = "✓" if r["correct"] else "✗"
        print(f"  id={r['id']:4d}  gold={r['gold']}  voted={voted!r}  {mark}")

## 9. Save Results

In [10]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 10 records to results/optimized_results.jsonl


## Next Steps

To run the full 1,126-question dataset: change `N_QUESTIONS = 10` → `N_QUESTIONS = len(data)` in the Configuration cell.

**If the logprobs calibration pass (`enable_thinking=False`) raises an error**: your version of `transformers` may not support it. Fallback: remove `make_calib_prompt` / Pass 4 and pass `calib_out=None` in the scoring call — the code will use text majority vote automatically.

**Further improvements:**
- Raise `N_SAMPLES_EXPAND` to 15–20 for very hard/ambiguous MCQ
- For free-form multi-part answers, parse each sub-answer individually and vote per slot
- Fine-tune on similar math datasets (permitted by competition rules)